In [7]:
#installing important libraries
#%pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_DIR = Path("")
RUL_CAP = 125

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Ready to begin.")
print("Dataset folder:", DATA_DIR.resolve())

Ready to begin.
Dataset folder: /Users/krishnashakya/Work/TATA_Elxsi_S4_PS1


In [ ]:
columns = (
    ["unit_number", "time_cycles", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
setting_cols = ["setting_1", "setting_2", "setting_3"]

train_path = DATA_DIR / "train_FD001.txt"
test_path = DATA_DIR / "test_FD001.txt"
rul_path = DATA_DIR / "RUL_FD001.txt"

for file in [train_path, test_path, rul_path]:
    if not file.exists():
        raise FileNotFoundError(
            f"Missing {file}. Put the three FD001 files in the folder"
            "next to this notebook, or change DATA_DIR."
        )

def load_cmapss(file_path):
    return pd.read_csv(file_path, sep=r"\s+", header=None, names=columns)

train_raw = load_cmapss(train_path)
test_raw = load_cmapss(test_path)
test_rul = pd.read_csv(rul_path, sep=r"\s+", header=None).iloc[:, 0].to_numpy()

print("Training data:", train_raw.shape)
print("Test data:", test_raw.shape)
print("Number of test RUL labels:", len(test_rul))
display(train_raw.head())

Training data: (20631, 26)
Test data: (13096, 26)
Number of test RUL labels: 100


,unit_number,time_cycles,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [9]:
print("Training engines:", train_raw["unit_number"].nunique())
print("Test engines:", test_raw["unit_number"].nunique())
print("Missing values:", int(train_raw.isna().sum().sum()))
print("Duplicate rows:", int(train_raw.duplicated().sum()))

display(train_raw.describe().T.round(2))

Training engines: 100
Test engines: 100
Missing values: 0
Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
unit_number,20631.0,51.51,29.23,1.00,26.00,52.00,77.00,100.00
time_cycles,20631.0,108.81,68.88,1.00,52.00,104.00,156.00,362.00
setting_1,20631.0,-0.00,0.00,-0.01,-0.00,0.00,0.00,0.01
setting_2,20631.0,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00
setting_3,20631.0,100.00,0.00,100.00,100.00,100.00,100.00,100.00
sensor_1,20631.0,518.67,0.00,518.67,518.67,518.67,518.67,518.67
sensor_2,20631.0,642.68,0.50,641.21,642.33,642.64,643.00,644.53
sensor_3,20631.0,1590.52,6.13,1571.04,1586.26,1590.10,1594.38,1616.91
sensor_4,20631.0,1408.93,9.00,1382.25,1402.36,1408.04,1414.55,1441.49
sensor_5,20631.0,14.62,0.00,14.62,14.62,14.62,14.62,14.62
